# 03 — Baseline models

> This system predicts glucose values for research and analysis. It is not a medical device and must not be used to determine insulin doses or make treatment decisions.

Rolling-origin validation results from `python -m src.train --cv`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, db, train
settings, mcfg = config.settings(), config.models()
duck = db.duck(read_only=True)
q = lambda sql: duck.execute(sql).df()

In [ ]:
cv = pd.read_parquet(config.path('data/predictions/cv_predictions.parquet'))
base = cv[cv.model.isin(['persistence', 'linear_trend', 'linear'])]
base.assign(ae=(base.y_pred - base.y_true).abs()).groupby(['model', 'target_type', 'horizon']).ae.mean().unstack().round(3)

## Per validation month (30 min)

In [ ]:
b30 = base[base.horizon == 30].assign(ae=lambda d: (d.y_pred - d.y_true).abs())
b30.groupby(['fold', 'model']).ae.mean().unstack().round(3)

## Why linear trend loses: residual vs recent rate (30 min)

In [ ]:
df, _ = train.load_dataset(settings)
lt = cv[(cv.model == 'linear_trend') & (cv.horizon == 30)].copy()
lt['ts'] = pd.to_datetime(lt.ts, utc=True)
lt['rate'] = df.loc[lt.ts, 'velocity_15m_mmol_per_h'].to_numpy()
lt['err'] = lt.y_pred - lt.y_true
lt.groupby(pd.cut(lt.rate, [-30, -6, -2, 2, 6, 30])).err.agg(['mean', 'count']).round(3)